In [1]:
import pandas as pd
import numpy as np
import joblib
import pyarrow.parquet as pq
import pyarrow as pa
import os
import time
from sklearn.ensemble import IsolationForest

print("Script started: Anomaly Detection using Isolation Forest.")
start_time = time.time()

# --- Configuration ---
INPUT_DATA_FILE = 'final_synthetic_dataset_full_dimensions.parquet'
OUTPUT_DATA_FILE = 'synthetic_data_with_anomalies.parquet'
MODEL_SAVE_PATH = 'isolation_forest_model.pkl'

# --- Training Configuration ---
# We will train on a sample of the data to be memory-efficient.
# 50,000 is a large sample that should provide a good model.
TRAINING_SAMPLE_SIZE = 50000 
# Contamination is the expected proportion of anomalies in the data. 
# 'auto' is a good default, but you can set it to a float like 0.01 (1%).
CONTAMINATION = 'auto' 

# --- Main Logic ---

def main():
    """
    Trains an Isolation Forest model on a sample of the data, then uses it
    to predict anomaly scores for the entire dataset in a memory-safe way.
    """
    if not os.path.exists(INPUT_DATA_FILE):
        print(f"Error: Input data file not found: {INPUT_DATA_FILE}")
        return

    try:
        parquet_file = pq.ParquetFile(INPUT_DATA_FILE)
        
        # --- 1. Train Model on a Sample ---
        print(f"\n--- Training Isolation Forest model on a sample of {TRAINING_SAMPLE_SIZE} rows ---")
        
        # Read the first row group to get a sample. This assumes the first chunk is large enough.
        # For more robust sampling, one could read multiple smaller chunks.
        sample_df = parquet_file.read_row_group(0).to_pandas()
        if len(sample_df) > TRAINING_SAMPLE_SIZE:
            sample_df = sample_df.sample(n=TRAINING_SAMPLE_SIZE, random_state=42)
        
        print(f"Training sample shape: {sample_df.shape}")

        # Initialize and train the Isolation Forest model
        iso_forest = IsolationForest(n_estimators=100, 
                                     contamination=CONTAMINATION, 
                                     random_state=42, 
                                     n_jobs=-1) # Use all available CPU cores
        
        iso_forest.fit(sample_df)
        print("Isolation Forest model has been trained successfully.")

        # Save the trained model for future use
        joblib.dump(iso_forest, MODEL_SAVE_PATH)
        print(f"Trained model saved to: {MODEL_SAVE_PATH}")


        # --- 2. Predict Anomalies on Full Dataset in Chunks ---
        print(f"\n--- Predicting anomalies on the full dataset in chunks ---")
        
        writer = None
        
        for i in range(parquet_file.num_row_groups):
            print(f"  Processing chunk {i+1}/{parquet_file.num_row_groups} for prediction...")
            
            chunk_df = parquet_file.read_row_group(i).to_pandas()
            
            # Predict anomaly scores. Lower scores are more anomalous.
            anomaly_scores = iso_forest.decision_function(chunk_df)
            
            # Predict anomaly labels (-1 for anomalies, 1 for inliers/normal)
            anomaly_labels = iso_forest.predict(chunk_df)
            
            # Add new columns to the chunk
            chunk_df['anomaly_score'] = anomaly_scores
            # Convert labels to a more intuitive boolean (True for anomaly)
            chunk_df['is_anomaly'] = (anomaly_labels == -1)
            
            # Convert to PyArrow Table
            table = pa.Table.from_pandas(chunk_df)
            
            # Initialize writer with the new, augmented schema
            if writer is None:
                writer = pq.ParquetWriter(OUTPUT_DATA_FILE, table.schema)
            
            writer.write_table(table)

        if writer:
            writer.close()
            
        print("\nPrediction complete.")
        print(f"Full dataset with anomaly scores saved to: {OUTPUT_DATA_FILE}")
        
        # --- 3. Display Anomaly Summary ---
        print("\n--- Anomaly Summary ---")
        # To get a summary, we can read just the new columns from the output file
        # without loading the whole dataset into memory.
        output_parquet = pq.ParquetFile(OUTPUT_DATA_FILE)
        is_anomaly_col = output_parquet.read(columns=['is_anomaly']).to_pandas()
        
        anomaly_count = is_anomaly_col['is_anomaly'].sum()
        total_count = len(is_anomaly_col)
        anomaly_percentage = (anomaly_count / total_count) * 100 if total_count > 0 else 0
        
        print(f"Total rows processed: {total_count}")
        print(f"Number of anomalies found: {anomaly_count}")
        print(f"Percentage of anomalies: {anomaly_percentage:.2f}%")


    except Exception as e:
        print(f"An error occurred during the process: {e}")

    end_time = time.time()
    total_minutes = (end_time - start_time) / 60
    print(f"\nTotal script execution time: {total_minutes:.2f} minutes.")


if __name__ == '__main__':
    main()


Script started: Anomaly Detection using Isolation Forest.
An error occurred during the process: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

Total script execution time: 0.00 minutes.
